In [1]:
import pandas as pd
import numpy as np
import joblib
import os

In [2]:
model = joblib.load(
    "../models/gradient_boosting_demand_model.pkl"
)

features = joblib.load(
    "../models/model_features.pkl"
)

metrics = joblib.load(
    "../models/model_metrics.pkl"
)

print("Model loaded successfully.")
print("\nFeatures:")
print(features)

print("\nMetrics:")
print(metrics)

Model loaded successfully.

Features:
['Monthly_Quantity', 'Lag_1', 'Lag_2', 'Lag_3', 'Rolling_3_Month', 'Rolling_6_Month', 'Monthly_Transactions', 'Average_Monthly_Price', 'Lag_1_Price', 'Price_Change_Pct', 'Log_Price_Ratio', 'Target_Month', 'Target_Quarter']

Metrics:
{'Model': 'Gradient Boosting - Corrected', 'MAE': 108.56746604496233, 'RMSE': np.float64(265.048677150999)}


In [5]:
monthly_demand = pd.read_csv(
    "../data/processed/monthly_demand_complete.csv"
)

monthly_demand["MonthYear"] = pd.PeriodIndex(
    monthly_demand["MonthYear"],
    freq="M"
)

print(monthly_demand.shape)
monthly_demand.head()

(79424, 18)


,StockCode,MonthYear,Monthly_Quantity,Monthly_Revenue,Monthly_Transactions,Average_Monthly_Price,Lag_1,Lag_2,Lag_3,Rolling_3_Month,Rolling_6_Month,Lag_1_Price,Price_Change_Pct,Price_Ratio,Log_Price_Ratio,Next_Month_Quantity,Month,Quarter
0,10002,2009-12,215.0,185.30,20.0,0.977500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,291.0,12,4
1,10002,2010-01,291.0,248.97,17.0,0.945294,215.0,NaN,NaN,NaN,NaN,0.977500,-3.294719,0.967053,-0.033502,257.0,1,1
2,10002,2010-02,257.0,220.07,15.0,0.958000,291.0,215.0,NaN,NaN,NaN,0.945294,1.344119,1.013441,0.013352,641.0,2,1
3,10002,2010-03,641.0,493.09,17.0,0.911111,257.0,291.0,215.0,254.333333,NaN,0.958000,-4.894456,0.951055,-0.050183,1132.0,3,1
4,10002,2010-04,1132.0,843.73,24.0,0.924167,641.0,257.0,291.0,396.333333,NaN,0.911111,1.432927,1.014329,0.014228,1465.0,4,2


In [7]:
print("Features saved in model_features.pkl:")
print(features)

print("\nFeatures expected by the model:")
print(model.feature_names_in_)

Features saved in model_features.pkl:
['Monthly_Quantity', 'Lag_1', 'Lag_2', 'Lag_3', 'Rolling_3_Month', 'Rolling_6_Month', 'Monthly_Transactions', 'Average_Monthly_Price', 'Lag_1_Price', 'Price_Change_Pct', 'Log_Price_Ratio', 'Target_Month', 'Target_Quarter']

Features expected by the model:
['Monthly_Quantity' 'Lag_1' 'Lag_2' 'Lag_3' 'Rolling_3_Month'
 'Rolling_6_Month' 'Monthly_Transactions' 'Average_Monthly_Price'
 'Lag_1_Price' 'Price_Change_Pct' 'Log_Price_Ratio' 'Target_Month'
 'Target_Quarter']


In [8]:
features = list(model.feature_names_in_)

print("Correct model features:")
print(features)

Correct model features:
['Monthly_Quantity', 'Lag_1', 'Lag_2', 'Lag_3', 'Rolling_3_Month', 'Rolling_6_Month', 'Monthly_Transactions', 'Average_Monthly_Price', 'Lag_1_Price', 'Price_Change_Pct', 'Log_Price_Ratio', 'Target_Month', 'Target_Quarter']


In [10]:
# Create next month's calendar features

monthly_demand["Target_Month"] = (
    monthly_demand["MonthYear"].dt.month % 12
) + 1

monthly_demand["Target_Quarter"] = (
    (monthly_demand["Target_Month"] - 1) // 3
) + 1

print(
    monthly_demand[
        ["MonthYear", "Target_Month", "Target_Quarter"]
    ].head(15)
)

   MonthYear  Target_Month  Target_Quarter
0    2009-12             1               1
1    2010-01             2               1
2    2010-02             3               1
3    2010-03             4               2
4    2010-04             5               2
5    2010-05             6               2
6    2010-06             7               3
7    2010-07             8               3
8    2010-08             9               3
9    2010-09            10               4
10   2010-10            11               4
11   2010-11            12               4
12   2010-12             1               1
13   2011-01             2               1
14   2011-02             3               1


In [11]:
model_data = monthly_demand.copy()

model_data = model_data.dropna(
    subset=["Next_Month_Quantity"]
).copy()

model_data_complete = model_data.dropna(
    subset=features
).copy()

print("Model data shape:", model_data_complete.shape)

Model data shape: (36265, 20)


In [12]:
missing_features = [
    col for col in features
    if col not in model_data_complete.columns
]

print("Missing features:", missing_features)

Missing features: []


In [13]:
missing_features = [
    col for col in features
    if col not in model_data_complete.columns
]

print("Missing features:", missing_features)

Missing features: []


In [14]:
train_data = model_data_complete[
    model_data_complete["MonthYear"] <= pd.Period("2011-06", freq="M")
].copy()

test_data = model_data_complete[
    model_data_complete["MonthYear"] >= pd.Period("2011-07", freq="M")
].copy()

print("Training shape:", train_data.shape)
print("Testing shape:", test_data.shape)

print(
    "\nTraining period:",
    train_data["MonthYear"].min(),
    "to",
    train_data["MonthYear"].max()
)

print(
    "Testing period:",
    test_data["MonthYear"].min(),
    "to",
    test_data["MonthYear"].max()
)

Training shape: (26182, 20)
Testing shape: (10083, 20)

Training period: 2010-06 to 2011-06
Testing period: 2011-07 to 2011-11


In [15]:
X_test = test_data[features]
y_test = test_data["Next_Month_Quantity"]

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_test shape: (10083, 13)
y_test shape: (10083,)


In [16]:
y_pred = model.predict(X_test)

# Demand cannot be negative
y_pred = np.maximum(y_pred, 0)

print("Predictions generated successfully.")
print("Number of predictions:", len(y_pred))
print("\nFirst 10 predictions:")
print(y_pred[:10])

Predictions generated successfully.
Number of predictions: 10083

First 10 predictions:
[ 83.3932609   79.9752902   91.58763832  72.07298594  61.73346196
  65.07727849  52.05811854  58.19861726 138.17254714  98.13180797]


In [17]:
prediction_results = test_data[
    ["StockCode", "MonthYear", "Next_Month_Quantity"]
].copy()

prediction_results["Predicted_Quantity"] = y_pred

prediction_results["Absolute_Error"] = (
    prediction_results["Next_Month_Quantity"]
    - prediction_results["Predicted_Quantity"]
).abs()

print(prediction_results.shape)
prediction_results.head(20)

(10083, 5)


,StockCode,MonthYear,Next_Month_Quantity,Predicted_Quantity,Absolute_Error
38,10080,2011-07,60.0,83.393261,23.393261
39,10080,2011-08,60.0,79.975290,19.975290
40,10080,2011-09,6.0,91.587638,85.587638
41,10080,2011-10,91.0,72.072986,18.927014
64,10120,2011-08,10.0,61.733462,51.733462
65,10120,2011-09,10.0,65.077278,55.077278
66,10120,2011-10,49.0,52.058119,3.058119
67,10120,2011-11,6.0,58.198617,52.198617
156,10125,2011-07,85.0,138.172547,53.172547
157,10125,2011-08,210.0,98.131808,111.868192


In [18]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(
    prediction_results["Next_Month_Quantity"],
    prediction_results["Predicted_Quantity"]
)

rmse = np.sqrt(
    mean_squared_error(
        prediction_results["Next_Month_Quantity"],
        prediction_results["Predicted_Quantity"]
    )
)

# MAPE only for non-zero actual demand
non_zero = prediction_results["Next_Month_Quantity"] != 0

mape = np.mean(
    np.abs(
        (
            prediction_results.loc[non_zero, "Next_Month_Quantity"]
            - prediction_results.loc[non_zero, "Predicted_Quantity"]
        )
        / prediction_results.loc[non_zero, "Next_Month_Quantity"]
    )
) * 100

print("Model Evaluation")
print("----------------")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)

Model Evaluation
----------------
MAE : 108.56488544265396
RMSE: 265.0478689356876
MAPE: 389.7201123266707


In [19]:
prediction_results.head(20)

,StockCode,MonthYear,Next_Month_Quantity,Predicted_Quantity,Absolute_Error
38,10080,2011-07,60.0,83.393261,23.393261
39,10080,2011-08,60.0,79.975290,19.975290
40,10080,2011-09,6.0,91.587638,85.587638
41,10080,2011-10,91.0,72.072986,18.927014
64,10120,2011-08,10.0,61.733462,51.733462
65,10120,2011-09,10.0,65.077278,55.077278
66,10120,2011-10,49.0,52.058119,3.058119
67,10120,2011-11,6.0,58.198617,52.198617
156,10125,2011-07,85.0,138.172547,53.172547
157,10125,2011-08,210.0,98.131808,111.868192


In [20]:
largest_errors = prediction_results.sort_values(
    "Absolute_Error",
    ascending=False
).head(20)

largest_errors

,StockCode,MonthYear,Next_Month_Quantity,Predicted_Quantity,Absolute_Error
7091,20971,2011-10,1389.0,7758.953156,6369.953156
27594,22197,2011-10,12452.0,6782.600912,5669.399088
25062,22086,2011-10,7898.0,3268.465711,4629.534289
62742,84879,2011-07,6545.0,2160.631928,4384.368072
56115,84077,2011-09,8084.0,3718.587406,4365.412594
7090,20971,2011-09,4872.0,634.763165,4237.236835
35402,22578,2011-10,5365.0,1876.107754,3488.892246
41226,22952,2011-08,3710.0,507.943911,3202.056089
810,16014,2011-10,3606.0,415.205878,3190.794122
27591,22197,2011-07,5421.0,2241.116024,3179.883976


In [21]:
prediction_results["Demand_Level"] = pd.cut(
    prediction_results["Next_Month_Quantity"],
    bins=[-1, 10, 50, 100, 500, 1000, 5000, np.inf],
    labels=[
        "0-10",
        "11-50",
        "51-100",
        "101-500",
        "501-1000",
        "1001-5000",
        "5000+"
    ]
)

error_by_level = (
    prediction_results
    .groupby("Demand_Level", observed=False)
    .agg(
        Count=("Next_Month_Quantity", "count"),
        MAE=("Absolute_Error", "mean"),
        Mean_Actual=("Next_Month_Quantity", "mean"),
        Mean_Predicted=("Predicted_Quantity", "mean")
    )
    .reset_index()
)

error_by_level

,Demand_Level,Count,MAE,Mean_Actual,Mean_Predicted
0,0-10,2414,38.564970,3.854184,42.419154
1,11-50,2701,46.245068,26.935579,71.904969
2,51-100,1327,58.407342,73.825923,119.143549
3,101-500,2638,117.237539,230.744503,244.463259
4,501-1000,645,316.108698,699.232558,591.293350
5,1001-5000,345,720.870764,1700.205797,1251.763389
6,5000+,13,2868.175329,6525.153846,3803.599279


In [22]:
prediction_results.to_csv(
    "../data/processed/next_month_demand_predictions.csv",
    index=False
)

print("Predictions saved successfully.")

Predictions saved successfully.
